In [0]:
--Organization Structure
  --electric_grid
    --data
      --cleaned_data table (parquet file)
      --volume (stored csv data)

CREATE CATALOG IF NOT EXISTS electric_grid;
USE CATALOG electric_grid;
CREATE SCHEMA IF NOT EXISTS electric_grid.data_schema;
USE SCHEMA data_schema;
CREATE VOLUME IF NOT EXISTS volume_set;


You'll need to upload the cleaned, database ready dataset and population dataset into the volume. After running the above SQL query, go to the Catalog tab near the top left (the symbol is three shapes) and open the electric_grid catalog, then data_schema schema, then volume_set volume. Highlight and click the three circles on the volume's tab and click *Upload to Volume*. Find the two datasets in your directory and upload. Use the code below to confirm it worked. 

In [0]:
--Parquet File
CREATE OR REPLACE TABLE cleaned_data AS
SELECT * FROM read_files(
  "/Volumes/electric_grid/data_schema/volume_set/doe_events_db_ready.csv",
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
-- Creates a table containing U.S states and respective population directly from excel file
CREATE OR REPLACE TABLE state_population AS
SELECT * FROM read_files(
  "/Volumes/electric_grid/data_schema/volume_set/NST-EST2025-POP.xlsx",
  format => 'excel',
  header => true,
  headerRows => 1,
  inferSchema => true
);

-- Remove . symbol from the State column
UPDATE state_population
SET State = REPLACE(State, ".", "")

In [0]:
SELECT * FROM state_population
LIMIT 5;

In [0]:
%python

#Parquet files offer efficient querying

# Compare CSV and Delta file sizes
csv_path =  "/Volumes/electric_grid/data_schema/volume_set/doe_events_db_ready.csv"

# Get CSV file size
csv_files = dbutils.fs.ls(csv_path)
csv_bytes = sum(f.size for f in csv_files if f.name.endswith('.csv'))

# Get Delta table size from DESCRIBE DETAIL
delta_size = spark.sql("DESCRIBE DETAIL cleaned_data").select("sizeInBytes").first()[0]

# Display comparison
print(f"CSV file size:    {csv_bytes:>15,} bytes  ({csv_bytes / 1024 / 1024:>8.1f} MB)")
print(f"Delta table size: {delta_size:>15,} bytes  ({delta_size / 1024 / 1024:>8.1f} MB)")
print(f"\nCompression ratio: {csv_bytes / delta_size:.1f}x smaller with Delta")

In [0]:
SELECT * FROM cleaned_data
LIMIT 5;

In [0]:
-- Create a table containin U.S states and respective population
-- to JOIN to cleaned_data and get affect customers of outage / population

SELECT * FROM cleaned_data cd
LEFT JOIN state_population sp
ON cd.area_affected_raw LIKE CONCAT('%', sp.State, '%');

-- The five columns after State are populations from 2020 to 2025, need to match that with relevant rows
  --Maybe create a seperate dataset with outages only from 2020 to 2025.


In [0]:
-- How many rows received a State column?
SELECT (SUM(CASE WHEN State IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS Null_Amount FROM cleaned_data cd
LEFT JOIN state_population sp
ON cd.area_affected_raw LIKE CONCAT('%', sp.State, '%');

-- 96%. There are probably false positives like "Kansas City" getting matched to "Kansas".

--